# Samsung PRISM 26VI05 - Gemma4 E2B comparison on Colab
1. Runtime -> Change runtime type -> **GPU**. E2B means 2.3B effective, 5.1B total parameters: test actual T4 memory first.
2. Runtime -> Run all. The model branch is cloned into `MyDrive/prism_ladder_gemma4`; images, checkpoints and reports
   are saved there too, so after a disconnect just Run all again and every stage resumes where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
branch = 'gemma4-model'
workdir = '/content/drive/MyDrive/prism_ladder_gemma4'
if not os.path.exists(workdir):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, 'https://github.com/Sanjana432/Samsung-prism-san.git', workdir], check=True)
os.chdir(workdir)
current = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
if current != branch:
    if current.rsplit('/', 1)[-1] != branch:
        raise RuntimeError('This folder uses a different model branch. Use its matching notebook/workspace.')
    subprocess.run(['git', 'branch', '-m', branch], check=True)
subprocess.run(['git', 'config', 'remote.origin.fetch', f'+refs/heads/{branch}:refs/remotes/origin/{branch}'], check=True)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'branch', '--set-upstream-to', f'origin/{branch}', branch], check=True)
subprocess.run(['git', 'merge', '--ff-only', f'origin/{branch}'], check=True)
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Native Gemma4 vision budget; separate checkpoints and predictions from other models.
import torch, yaml
assert torch.cuda.is_available(), 'Choose a GPU runtime before running model stages.'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
assert cfg['model']['base'] == 'google/gemma-4-E2B-it'
if mem < 20:
    cfg['inference']['batch_size'] = 1
    with open('configs/config.yaml', 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
print(f'{mem:.0f} GB -> soft token budget', cfg['model']['max_soft_tokens'], 'batch', cfg['inference']['batch_size'])
print('Run the GPU smoke test before long jobs; full Gemma4 E2B GPU memory is not just 2B parameters.')

In [ ]:
import json
if not os.path.exists('data/metadata/ladder.jsonl'):
    !python scripts/01_generate_dataset.py
!python scripts/02_build_splits.py
if not os.path.exists('pilot/images'):
    !python scripts/make_pilot_showcase.py
!python scripts/00_selftest.py

In [ ]:
!python scripts/gpu_smoke_test.py

In [ ]:
!python scripts/03_prompt_study_and_baseline.py

In [ ]:
!python scripts/04_train_qlora.py

In [ ]:
!python scripts/05_infer.py --adapter outputs/best_adapter --name finetuned_qlora --split test
!python scripts/06_evaluate.py --models baseline_zero_shot finetuned_qlora --split test

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('reports/evaluation_report.md').read()))
display(Image('reports/figures/mae_per_metric.png'))